In [ ]:
import os

os.environ.pop("CPU", None)
os.environ.pop("ACCELERATE_USE_CPU", None)

import requests
import pandas as pd
import torch
from sklearn.model_selection import train_test_split
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    TrainingArguments,
    Trainer
)

DATA_URLS = [
    "https://raw.githubusercontent.com/sanskrit-sandhi/SandhiKosh/master/Astaadhyaayii%20Corpus.xls",
    "https://raw.githubusercontent.com/sanskrit-sandhi/SandhiKosh/master/Bhagvad_Gita%20Corpus.xls",
    "https://raw.githubusercontent.com/sanskrit-sandhi/SandhiKosh/master/Rule-based%20Corpus%20and%20Literature%20Corpus.xls",
    "https://raw.githubusercontent.com/sanskrit-sandhi/SandhiKosh/master/UoH_Corpus.xls"
]
DATA_FILES = [
    "Astaadhyaayii_Corpus.xls",
    "Bhagvad_Gita_Corpus.xls",
    "Rule_based_Corpus.xls",
    "UoH_Corpus.xls"
]
MODEL_NAME = "google/byt5-base"
OUTPUT_DIR = "./sanskrit_sandhi_model"

In [ ]:
def download_dataset():
    for url, filename in zip(DATA_URLS, DATA_FILES):
        if not os.path.exists(filename):
            print(f"Downloading {filename}...")
            r = requests.get(url)
            with open(filename, "wb") as f:
                f.write(r.content)
    print("Datasets ready.")


def load_dataset():
    dfs = []
    for filename in DATA_FILES:
        try:
            df = pd.read_excel(filename)
            if 'Word' in df.columns and 'Split' in df.columns:
                df = df[['Word', 'Split']].rename(columns={"Word": "input", "Split": "split"})
            elif 'compound' in df.columns and 'split' in df.columns:
                 df = df[['compound', 'split']].rename(columns={"compound": "input", "split": "split"})
            else:
                print(f"Error in {filename}. {df.columns}")
                continue

            dfs.append(df)
        except Exception as e:
            print(f"err {filename}: {e}")

    full_df = pd.concat(dfs, ignore_index=True)
    return full_df

In [ ]:
def format_split(split):
    if not isinstance(split, str):
        return str(split)
    parts = split.replace(" ", "").split("+")
    return "&" + "+".join(parts) + "$"


def preprocess_dataframe(df):
    df = df.dropna(subset=["input", "split"])
    df["target"] = df["split"].apply(format_split)
    df =df[["input", "target"]]

    return df


def create_datasets(df):
    train_df, test_df = train_test_split(df, test_size=0.1, random_state=42)

    train_dataset = Dataset.from_pandas(train_df)
    test_dataset = Dataset.from_pandas(test_df)

    return train_dataset, test_dataset, test_df

In [ ]:
def tokenize_dataset(train_dataset, test_dataset, tokenizer):


    def preprocess(example):
        inputs = tokenizer(example["input"], max_length=256, truncation=True, padding="max_length")
        labels = tokenizer(example["target"], max_length=256, truncation=True, padding="max_length")

        # -100
        labels["input_ids"] = [
            (l if l != tokenizer.pad_token_id else -100)
            for l in labels["input_ids"]
        ]

        inputs["labels"] = labels["input_ids"]
        return inputs

    train_dataset = train_dataset.map(preprocess)
    test_dataset = test_dataset.map(preprocess)

    return train_dataset, test_dataset

In [ ]:
def train_model(train_dataset, test_dataset, tokenizer):

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)

    training_args = TrainingArguments(
        output_dir=OUTPUT_DIR,
        # learning_rate=0.00005,
        learning_rate=0.0001,
        per_device_train_batch_size=8,
        gradient_accumulation_steps=2,
        per_device_eval_batch_size=8,
        # num_train_epochs=12
        num_train_epochs=10,
        eval_strategy="epoch",
        save_strategy="epoch",
        logging_steps=50,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss"
    )

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=train_dataset,
        eval_dataset=test_dataset,
        processing_class=tokenizer,
    )

    trainer.train()

    return model

In [ ]:
from tqdm.auto import tqdm

def predict(model, tokenizer, word):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device)
    inputs = tokenizer(word, return_tensors="pt")
    inputs = {k: v.to(device) for k, v in inputs.items()}

    # 256
    outputs = model.generate(**inputs, max_new_tokens=256)
    result = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return result


def evaluate(model, tokenizer, test_df):
    correct = 0
    total = len(test_df)

    for _, row in tqdm(test_df.iterrows(), total=total):
        pred = predict(model, tokenizer, row["input"])
        if pred == row["target"]:
            correct += 1

    accuracy = correct / total
    print(f"Segmentation Accuracy: {accuracy:.4f}")


def evaluate_show_samples(model, tokenizer, test_df, n=10):
    correct = 0
    total = len(test_df)
    samples = []

    for _, row in tqdm(test_df.iterrows(), total=total):
        pred = predict(model, tokenizer, row["input"])
        if pred == row["target"]:
            correct += 1
        if len(samples) < n:
            samples.append((row["input"], row["target"], pred))

    accuracy = correct / total
    print(f"\nSegmentation Accuracy: {accuracy:.4f}\n")
    print("Sample (input -> target | pred):")
    for inp, tgt, pr in samples:
        match = "good" if tgt == pr else "bad"
        print(f"  {match} {inp[:40]!r} -> {tgt[:50]!r} | {pr[:50]!r}")


def demo_predictions(model, tokenizer):
    test_words = [
        "narendra",
        "tatrApi",
        "devadatta",
        "mahAtmA"
    ]

    print("\nExample predictions:\n")

    for word in test_words:
        print(word, "->", predict(model, tokenizer, word))

In [ ]:
download_dataset()
df = load_dataset()
df.head()

Datasets ready.
Skipping Rule_based_Corpus.xls: unexpected columns Index(['Unnamed: 0', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4'], dtype='object')


,input,split
0,वृद्धिरादैच्,वृद्धिः+आदैच्
1,इको गुणवृद्धी,इकः+गुणवृद्धी
2,न धातुलोप आर्धधातुके,न+धातुलोपे+आर्धधातुके
3,ग्क्ङिति च,क्क्ङिति+च
4,हलोऽनन्तराः संयोगः,हलः+अनन्तराः+संयोगः


In [ ]:
#Preprocessing
df = preprocess_dataframe(df)
train_dataset, test_dataset, test_df = create_datasets(df)
print(f"Train dataset size: {len(train_dataset)}")
print(f"Test dataset size: {len(test_dataset)}")

Train dataset size: 12143
Test dataset size: 1350


/tmp/ipykernel_620/2496014161.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df["target"] = df["split"].apply(format_split)


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

train_dataset, test_dataset = tokenize_dataset(
    train_dataset,
    test_dataset,
    tokenizer
)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/721 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/12143 [00:00<?, ? examples/s]

Map:   0%|          | 0/1350 [00:00<?, ? examples/s]

In [ ]:
# Training
import torch
print(f"Training on: {'GPU' if torch.cuda.is_available() else 'CPU'}")

model = train_model(train_dataset, test_dataset, tokenizer)

Training on: GPU (CUDA)


pytorch_model.bin:   0%|          | 0.00/2.33G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/254 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to encoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
The tied weights mapping and config for this model specifies to tie shared.weight to decoder.embed_tokens.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning


generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [ ]:
from tqdm.auto import tqdm

correct = 0
total = len(test_df)

for _, row in tqdm(test_df.iterrows(), total=total):
    pred = predict(model, tokenizer, row["input"])
    if pred == row["target"]:
        correct += 1

accuracy = correct / total
print(f"Segmentation Accuracy: {accuracy:.4f}")

Evaluating with progress bar...


  0%|          | 0/1350 [00:00<?, ?it/s]

Segmentation Accuracy: 0.8467


In [ ]:
demo_predictions(model, tokenizer)


Example predictions:

narendra → &त्+इिम्+अिः+इिः+इिः+इिः+इिः+इिः+इिः+इिः+इिः+इिः+इिः+इिः+इिः+िः+इिः+िः+इिः+िः+इिः+िः+इिः+िः+इिः+
tatrApi → &अपि+अरि$
devadatta → &अस्थाने+अस्थाने+अस्थाने$
mahAtmA → &त्त्त्त्त्त्त्त्त्त्त्त्त्त्त्त्त्त्त्त्त्त्त्त्त्त्त्त्त्त्त्त्त्त्त्त्त्त्त्त्त्त्त
